In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
import torch
import matplotlib.pyplot as plt 
from simpledesign.models.simpledesign import SimpleDesign
from safetensors.torch import load_file
from pathlib import Path

**Loading the Model**

Since we initialize the sequence track using ESM2 weights, we have to build the model using parameters that match the one used for ESM2. In this case we are loading a 8M version that needs:

- model dimension: 320
- num_heads: 20
- number of blocks: 6
- MLP dimension: 1280 (4*model_d)

By doing this we are initializing the sequence track. 

In [9]:
# load the model from a version of esm2: 
model = SimpleDesign(model_d=320, n_blocks=6, num_heads=20, mlp_d=1280, sigma=0.1)
model.load_esm2(load_file("/Users/thomasbush/Documents/ML/simpledesign/checkpoints/esm2_t6_8M_UR50D/model.safetensors"))

**Forward Pass**

To make a forward pass the model needs: 

- Sequences (B, L, )
- Coordinates (B, L, 3) + <cls>, <eos> pad slots 
- Sequence Mask real tokens
- Structure Mask  with pos 0 and l-1 for the special slots + pad
- idx: arange(L)-> the idx for each component in the sequence
- t and t' (B,)

## Testing the Model


To test the model we are going to load a sequence + structure

- Build the parsers
- Create a simple data loader -> convert to the right dims (add noise later)
- Do a forward pass